In [1]:
import pandas as pd

In [27]:
def df_transform(df: pd.DataFrame) -> pd.DataFrame:
    """Transpose week-columns into planned gate columns.

    Input: raw Sheet1 DataFrame with a 'Work Package' column and integer week
    columns (1–52) whose cells contain stage gate numbers 1–4.
    Output: one row per work package with planned_gate_1..4 and
    forecast_gate_1..4 (all NULL placeholders).
    """
    week_columns = [col for col in df.columns if isinstance(col, (int, float)) and not pd.isna(col)]
    week_columns = [int(c) for c in week_columns]

    transformed_data = []

    for _, row in df.iterrows():
        work_package = row.get("Work Package", "")
        stage_gate_dict = {"work_package_name": work_package}

        for week in week_columns:
            val = row.get(week)
            if val in [1, 2, 3, 4]:
                key = f"planned_gate_{int(val)}"
                if key not in stage_gate_dict:
                    stage_gate_dict[key] = week

        transformed_data.append(stage_gate_dict)

    df_transformed = pd.DataFrame(transformed_data)

    # Keep planned columns in order
    planned_cols = [
        "work_package_name",
        "planned_gate_1",
        "planned_gate_2",
        "planned_gate_3",
        "planned_gate_4",
    ]
    df_transformed = df_transformed[[c for c in planned_cols if c in df_transformed.columns]]

    # Append forecast gate columns with SQL-friendly NULL values
    for i in range(1, 5):
        df_transformed[f"forecast_gate_{i}"] = None

    return df_transformed

In [28]:
excel_path = '../data/client_provided_data/MASTER 2026 Delivery Programme.xlsx'
sheet_name = 'Sheet1'
df = pd.read_excel(excel_path, sheet_name=sheet_name)
df = df_transform(df)
df.head(5)

,work_package_name,planned_gate_1,planned_gate_2,planned_gate_3,planned_gate_4,forecast_gate_1,forecast_gate_2,forecast_gate_3,forecast_gate_4
0,N Yorks - 01,11.0,27.0,28.0,44.0,None,None,None,None
1,N Yorks - 02,11.0,33.0,34.0,NaN,None,None,None,None
2,N Yorks - 03,11.0,39.0,40.0,NaN,None,None,None,None
3,Blackburn & Darwen - 01,NaN,9.0,15.0,30.0,None,None,None,None
4,Blackburn & Darwen - 02,1.0,14.0,20.0,33.0,None,None,None,None


In [ ]:
# df_transformed.to_excel('../data/stage_gate_transformed.xlsx', index=False)